In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import svm
from sklearn.metrics import f1_score
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from scipy.special import softmax

In [2]:
print(f"Cargando datos...")
dataset_train = pd.read_json("./data/dataset_polaridad_es_train_embeddings.json", lines=True)
dataset_test = pd.read_json("./data/dataset_polaridad_es_test_embeddings.json", lines=True)


Cargando datos...


In [ ]:
LEncoder = LabelEncoder()
# El texto ya está en su forma vectorial calculado por los word embeddings del archivo según el campo
# Campo = Vector de Word Embeddings de 300 dimensiones
# we_ft = Word Embeddings calculados de textos generales del español 
# we_mx = Word Embeddings calculados de textos del español de México
# we_es = Word Embeddings calculados de textos del español de España

X_trainext = dataset_train['text'].to_numpy()
# Convertir a matriz de arrays de numpy
X_es = np.vstack(dataset_train["we_es"].to_numpy())
X_mx = np.vstack(dataset_train["we_mx"].to_numpy())
X_ft = np.vstack(dataset_train["we_ft"].to_numpy())
Y_train = dataset_train['klass'].to_numpy()

X_train = np.concatenate([X_es, X_mx, X_ft], axis=1) 


In [20]:

X_test_text = dataset_test['text'].to_numpy()
# Convertir a matriz de arrays de numpy
#X_es_t = np.vstack(dataset_test["we_es"].to_numpy())
#X_mx_t = np.vstack(dataset_test["we_mx"].to_numpy())
X_ft_t = np.vstack(dataset_test["we_ft"].to_numpy())
X_test = X_ft_t
Y_test = dataset_test['klass'].to_numpy()

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Y_train_encoded= LEncoder.fit_transform(Y_train)
Y_test_encoded= LEncoder.transform(Y_test)

In [21]:
X_train, X_val, Y_train_encoded, Y_val_encoded = train_test_split(
    X_train,
    Y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=Y_train_encoded
)

In [28]:
from torch import nn
import numpy as np
# Definir la red neuronal en PyTorch heredando de la clase base de Redes Neuronales: Module
class RedNeuronal(nn.Module):
    def __init__(self, input_size, output_size,c1,c2,c3):
        super(RedNeuronal, self).__init__()
        
        input_size_h1 = c1
        input_size_h2 = c2
        input_size_h3 = c3

        # Capa 1: Recibe muchas características
        self.fc1 = nn.Linear(input_size, input_size_h1)
        self.bn1 = nn.BatchNorm1d(input_size_h1)
        self.act1 = nn.LeakyReLU(0.01)
        self.drop1 = nn.Dropout(0.5)
        
        # Capa 2: Compresión
        self.fc2 = nn.Linear(input_size_h1, input_size_h2)
        self.bn2 = nn.BatchNorm1d(input_size_h2)
        self.act2 = nn.LeakyReLU(0.01)
        self.drop2 = nn.Dropout(0.5)

        # Capa 3: Refinamiento
        self.fc3 = nn.Linear(input_size_h2, input_size_h3)
        self.bn3 = nn.BatchNorm1d(input_size_h3)
        self.act3 = nn.LeakyReLU(0.01)
        self.drop3 = nn.Dropout(0.3)

        # Salida
        self.output = nn.Linear(input_size_h3, output_size)
        
    def forward(self, x):
        x = self.drop1(self.act1(self.bn1(self.fc1(x))))
        x = self.drop2(self.act2(self.bn2(self.fc2(x))))
        x = self.drop3(self.act3(self.bn3(self.fc3(x))))
        x = self.output(x)
        return x

In [31]:
import torch
import numpy as np
import random
from sklearn.metrics import f1_score


def inicializar_poblacion(tam_poblacion, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion):
    # Cada individuo es del tamaño del total de los pesos y bias que conforman a la red neuronal
    # Un individuo está formado por los pesos y bias de todas sus capas en forma de un vector
    # indviduo = [w11(1), w12(1), w13(1), b1(1), b2(1), w11(2), w12(2, b1(2), b2(2), ....]
    # w11(1) representa el peso w11 de la capa 1
    # wij(n) representa un peso i,j de la capa n
    # bk(n) representa un sesgo (bias) k de la capa n
    
    red = RedNeuronal(tam_entrada, tam_capa_oculta, tam_capa_oculta, tam_salida, tasa_reduccion)    
    tam_individuo = red.tam_individuo
    poblacion = np.random.uniform(low=-0.5, high=0.5, size=(tam_poblacion, tam_individuo))
    return poblacion
    

def ajustar_estructura_red_neuronal(individuo, red_neuronal, tam_entrada, tam_capa_oculta, tam_salida):
 # Cargar pesos del individuo
    with torch.no_grad():
        # La matriz de pesos en Pytorch tienen la forma transpuesta (salida, entradas) a la inversa como se define la capa Lineal
        # Se reestablece el vector de pesos y bias que representa a cada individuo a su estrcutura de la red neuronal
        # 1. Se extraen las secciones (slice) del individuo (vector) que corresponden a la capa1 (fc1) y se extraen los bias de la capa1 (fc1)
        # 1.2. Se asignan a las secciones correspondiente de la red (weight.data) y (bias.data). 
        # 1.3.  Deben tener la misma forma (shape) que la estructura de la red definida, de lo contrario indicará el error.
        # 2. Se repiten el proceso para las capas restantes: desplazándose la sección de los pesos y bias de la primera capa (fc1) y extraer
        #    los pesos y bias de la capa2 (fc2) y asignarlos a los parámetros correspondientes.
        red_neuronal.fc1.weight.data = torch.tensor(individuo[:red_neuronal.tam_capa_1_pesos].reshape(tam_capa_oculta, tam_entrada)).float()
        desplazamiento_capa_1 =   red_neuronal.tam_capa_1_pesos + red_neuronal.tam_capa_1_bias
        red_neuronal.fc1.bias.data = torch.tensor(individuo[red_neuronal.tam_capa_1_pesos:desplazamiento_capa_1]).float()
        
        red_neuronal.fc2.weight.data = torch.tensor(individuo[desplazamiento_capa_1:desplazamiento_capa_1 + red_neuronal.tam_capa_2_pesos].reshape(red_neuronal.tam_capa_oculta2, tam_capa_oculta)).float()
        desplazamiento_capa_2 = desplazamiento_capa_1 + red_neuronal.tam_capa_2_pesos + red_neuronal.tam_capa_2_bias
        red_neuronal.fc2.bias.data = torch.tensor(individuo[desplazamiento_capa_1 + red_neuronal.tam_capa_2_pesos:desplazamiento_capa_2]).float()
        
        red_neuronal.fc3.weight.data = torch.tensor(individuo[desplazamiento_capa_2:desplazamiento_capa_2 + red_neuronal.tam_capa_3_pesos].reshape(tam_salida, red_neuronal.tam_capa_oculta2)).float()
        desplazamiento_capa_3 = desplazamiento_capa_2 + red_neuronal.tam_capa_3_pesos + red_neuronal.tam_capa_3_bias
        red_neuronal.fc3.bias.data = torch.tensor(individuo[desplazamiento_capa_2 + red_neuronal.tam_capa_3_pesos:desplazamiento_capa_3]).float()
    return red_neuronal

#  Función para evaluar el individuo (que representa su genoma)
def funcion_fitness(individuo, X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion):
    red_neuronal = RedNeuronal(tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
    red_neuronal = ajustar_estructura_red_neuronal(individuo, red_neuronal, tam_entrada, tam_capa_oculta, tam_salida)

    # Calcular precisión

    X_tensor = torch.tensor(X).float()

    with torch.no_grad():

        y_pred = red_neuronal(X_tensor)
        # Obtiene una única clase, la más probable
        y_pred = torch.argmax(y_pred, dim=1)
        score = f1_score(Y, y_pred, average="macro")
            
    return score



def evaluar_fitness(poblacion,  X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion):
    fitness = []
    for individuo in poblacion:
        val_fitness = funcion_fitness(individuo, X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
        fitness.append(val_fitness)
    return  np.array(fitness)

    
#------------------------------------------------------------------------
def seleccionar_padres(poblacion, aptitudes):
    """Selecciona dos padres mediante torneo."""
    torneo = random.sample(list(zip(poblacion, aptitudes)), k=4)
    torneo.sort(key=lambda x: x[1])  
    return torneo[0][0], torneo[1][0]

#------------------------------------------------------------------------
def elitismo(poblacion, fitness, tam_elite=2):
    """Selecciona los 'tam_elite' mejores best_individuos."""
    # Ordenar por fitness (mayor = mejor)
    ranked_indices = np.argsort(fitness)[::-1]
    elites = [poblacion[i] for i in ranked_indices[:tam_elite]]
    return elites


#------------------------------------------------------------------------
def cruzar(padre1, padre2):
    punto_cruza = np.random.randint(len(padre1))
    hijo1 = np.concatenate([padre1[:punto_cruza], padre2[punto_cruza:]])
    hijo2 = np.concatenate([padre2[:punto_cruza], padre1[punto_cruza:]])
    return hijo1, hijo2

#------------------------------------------------------------------------
def mutar(individuo, tasa_mutacion=0.1):
    # Crea la mascara de genes por mutar que cumplan con la condición
    mascara = np.random.rand(len(individuo)) < tasa_mutacion
    individuo = mascara * np.random.normal(0, 1, size=len(individuo))  # Mutar al individuo en los genes seleccionados 
    return individuo


    
def algoritmo_evolutivo(X, Y, X_val, Y_val, tam_poblacion=30, num_generaciones=50):
    tam_entrada = X.shape[1] # Características TF-IDF, columnas
    tam_capa_oculta = 128
    tam_salida = 3  # 3 categorías
    tasa_reduccion = 0.8
        
    # Población inicial
    poblacion = inicializar_poblacion(tam_poblacion, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
    best_fitness_hist = []
    mean_fitness_hist = []

    for generacion in range(num_generaciones):
        # Evaluar fitness (usando el conjunto de entrenamiento)
        val_fitness = evaluar_fitness(poblacion,  X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
        
        # Registrar estadísticas
        print("obteniendo mejor fitness" )
        best_fitness = np.max(val_fitness)
        print("obteniendo media fitness " )
        mean_fitness = np.mean(val_fitness)
        best_fitness_hist.append(best_fitness)
        mean_fitness_hist.append(mean_fitness)
        
        print(f"Generación {generacion + 1}: Mejor fitness = {best_fitness:.4f}, Fitness promedio = {mean_fitness:.4f}")
        nueva_poblacion = []

        for _ in range(tam_poblacion // 2):
            # Seleccionar padres
            padre1, padre2 = seleccionar_padres(poblacion, val_fitness)            
            # Crear descendencia mediante cruce
            hijo1, hijo2 = cruzar(padre1, padre2)
            hijo1 = mutar(hijo1)
            hijo2 = mutar(hijo2)
            nueva_poblacion.append(hijo1)            
            nueva_poblacion.append(hijo2)        
        
        
        #-----------------
        # Población: Los hijos sustituyen a los padres
        #-----------------
        # poblacion = np.array(nueva_poblacion)
    
        #-----------------
        # Población con elitismo de padres y parte de los hijos
        #-----------------
        nueva_poblacion = np.array(nueva_poblacion)
        K_best_padres = 5
        poblacion[:K_best_padres, ] = elitismo(poblacion, val_fitness, K_best_padres)
        poblacion[K_best_padres:, ] = nueva_poblacion[K_best_padres:, ]


    val_fitness = evaluar_fitness(poblacion,  X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)

    # Evaluar el mejor modelo en train
    best_individuo = poblacion[np.argmax(val_fitness)]
    test_f1 = funcion_fitness(best_individuo, X, Y, X_val, Y_val, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
    print(f"\nF1-score final en test: {test_f1:.3f}")
    return best_individuo


def predecir_clase(individuo, X, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion):
    red_neuronal = RedNeuronal(tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)
    red_neuronal = ajustar_estructura_red_neuronal(individuo, red_neuronal, tam_entrada, tam_capa_oculta, tam_salida)
    
    

    # Calcular precisión
    X_tensor = torch.tensor(X).float()

    with torch.no_grad():
        # Calcula las predicciones con la red neuronal con los pesos y bias definidos por el individuo
        y_pred = red_neuronal(X_tensor)
        y_pred = torch.argmax(y_pred, dim=1)
        return y_pred

In [32]:

best_individuo = algoritmo_evolutivo(X_train, Y_train_encoded, X_val, Y_val_encoded, tam_poblacion=50, num_generaciones=20)

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [25]:
import numpy as np
import pandas as pd

tam_entrada = len(X_test[0])
tam_capa_oculta = 128
tam_salida = 3  # 3 categorías
tasa_reduccion = 0.8

y_pred_test = predecir_clase(best_individuo, X_test, tam_entrada, tam_capa_oculta, tam_salida, tasa_reduccion)

score = f1_score(Y_test_encoded, y_pred_test, average="macro")
print(f"\nF1-score final en test: {score:.3f}")




F1-score final en test: 0.157
